---
title: "2.19. Zapytania tekstowe i wyrażenia regularne (REGEX)"
subtitle: Najpierw przykład, potem własny wzorzec
---

# 2.19. Zapytania tekstowe i wyrażenia regularne (REGEX)

W danych tekstowych często szukamy określonej **postaci zapisu**: kodu produktu, numeru grupy, rozszerzenia pliku albo kwoty. Wyrażenie regularne, czyli REGEX, opisuje taki wzorzec. Pozwala wskazać pasujące napisy, wydobyć fragmenty i zastąpić je innym tekstem.

Ten podrozdział rozwija podstawy funkcji tekstowych z rozdziału 2.11. Pracujemy w cyklu **objaśnienie → przykład pokazowy → wynik i interpretacja → zadanie**. Wszystkie przykłady korzystają z bazowego R i danych utworzonych tutaj. Uruchamiaj komórki po kolei. Wyniki pokazów są zapisane w notebooku, więc można je czytać bez wykonywania kodu.

Do eksperymentów użyjemy również [RegExr](https://regexr.com/), który pokazuje dopasowania i objaśnia elementy wzorca. Serwis obsługuje silniki JavaScript i PHP/PCRE. Nie wykonuje kodu R: wzorzec sprawdzony w przeglądarce trzeba jeszcze uruchomić na docelowych danych w R.

## 1. Pierwszy pokaz: znajdź fragment tekstu

Na początek szukamy dosłownego fragmentu `raport`. `grepl()` zwraca po jednej odpowiedzi logicznej dla każdego napisu. `grep()` domyślnie podaje pozycje pasujących napisów, a z `value = TRUE` same napisy.

In [1]:
pliki <- c("raport_01.csv", "notatki.txt", "RAPORT_02.CSV", "stary_raport.csv")
grepl("raport", pliki)
grep("raport", pliki)
grep("raport", pliki, value = TRUE, ignore.case = TRUE)

[1]  TRUE FALSE FALSE  TRUE
[1] 1 4
[1] "raport_01.csv"    "RAPORT_02.CSV"    "stary_raport.csv"


Pierwszy wynik to `TRUE, FALSE, FALSE, TRUE`: wielkość liter ma znaczenie. Po użyciu `ignore.case = TRUE` znajdziemy również trzeci plik. Wzorzec nie wymaga, aby słowo znajdowało się na początku napisu, więc pasuje też `stary_raport.csv`.

**Zadanie R.1.** Utwórz wektor `c("ankieta.csv", "ANKIETA.xlsx", "wyniki.csv", "stara_ankieta.txt")`. Znajdź wszystkie nazwy zawierające `ankieta` bez rozróżniania wielkości liter. Pokaż osobno pozycje oraz pasujące nazwy. Oczekiwane są trzy nazwy.

## 2. Pokaz w RegExr: od wzorca do podświetlenia

Otwórz [regexr.com](https://regexr.com/). W polu **Expression** ustaw treść wzorca `raport_[0-9]{2}\.csv`. W polu **Text** wklej poniższe fikcyjne nazwy, każdą w osobnym wierszu:

```text
raport_01.csv
raport_12.csv
raport_123.csv
RAPORT_02.CSV
raport_ab.csv
```

Wybierz silnik JavaScript i flagę `g`, aby obejrzeć wszystkie dopasowania. Wzorzec powinien zaznaczyć pierwsze dwa wiersze. Po włączeniu flagi `i` zaznaczy również czwarty. Najedź na `[0-9]` i `{2}` lub skorzystaj z objaśnienia **Explain**, aby sprawdzić ich rolę. W trybie **Tests** można dodać przypadki oczekiwanego dopasowania i jego braku. W tym ćwiczeniu testujemy dane przykładowe; nie trzeba ich zapisywać ani publikować.

Teraz wykonaj ten sam pokaz w R:

In [2]:
nazwy_demo <- c("raport_01.csv", "raport_12.csv", "raport_123.csv",
                "RAPORT_02.CSV", "raport_ab.csv")
grep("raport_[0-9]{2}\\.csv", nazwy_demo,
     ignore.case = TRUE, perl = TRUE, value = TRUE)

[1] "raport_01.csv" "raport_12.csv" "RAPORT_02.CSV"


Otrzymasz nazwy pierwszą, drugą i czwartą. `[0-9]{2}` oznacza dokładnie dwie cyfry, a `\.` dosłowną kropkę. W napisie R potrzebne są **dwa ukośniki odwrotne**, ponieważ pierwszy służy zapisowi znaku `\` w łańcuchu. Do R nie kopiujemy otaczających wzorzec delimiterów `/.../g`.

**Zadanie R.2.** Zmień wymagane dwie cyfry na trzy. Przewidź wynik w RegExr, a potem sprawdź w R. Dopisz `kopia_raport_123.csv` i zobacz, czy również pasuje. Dlaczego?

## 3. Dwa poziomy zapisu: REGEX i napis R

Zanim silnik dopasuje wzorzec, R musi odczytać łańcuch znaków. Dlatego `"\\d"` w kodzie R przekazuje do silnika wzorzec `\d`. Poniżej jawnie używamy `perl = TRUE` dla wyrażeń regularnych, aby zachować jeden tryb pracy, także w późniejszych przykładach z grupowaniem i sprawdzaniem kontekstu.

| Cel | Wzorzec wpisany w RegExr | Łańcuch w kodzie R |
| :--- | :--- | :--- |
| Dosłowna kropka | `\.` | `"\\."` |
| Co najmniej jedna cyfra | `\d+` | `"\\d+"` |
| Co najmniej jeden biały znak | `\s+` | `"\\s+"` |
| Całe słowo `kot` | `\bkot\b` | `"\\bkot\\b"` |
| Dosłowny ukośnik odwrotny | `\\` | `"\\\\"` |

In [3]:
wzorzec_cyfr <- "\\d+"
print(wzorzec_cyfr)  # reprezentacja napisu w R
cat(wzorzec_cyfr, "\n")  # treść przekazywana do silnika

grepl(wzorzec_cyfr, c("grupa 12", "grupa A"), perl = TRUE)

[1] "\\d+"
\d+ 
[1]  TRUE FALSE


Nie utożsamiaj `perl = TRUE` z gwarancją identycznego działania wszystkich wzorców w RegExr. JavaScript, PHP/PCRE i silnik używany w R mogą różnić się wersją oraz obsługą Unicode i opcji. Podstawy trybów pracy opisuje [dokumentacja REGEX w R](https://stat.ethz.ch/R-manual/R-devel/library/base/html/regex.html). Szczególnie ostrożnie przenoś granice słów i klasy znaków przy polskich literach; `[A-Za-z]` nie obejmuje np. `ą` ani `ł`.

**Zadanie R.3.** Zapisz w R wzorzec `\s+` i pokaż go przez `cat()`. Sprawdź `c("Jan Kowalski", "Jan_Kowalski", "Jan\tKowalski")`. Które napisy zawierają biały znak?

## 4. Kropka, znaki specjalne i wyszukiwanie dosłowne

Kropka we wzorcu zastępuje pojedynczy znak (w używanym trybie PCRE domyślnie poza znakiem nowej linii). Dlatego `.csv` może znaleźć coś innego niż rozszerzenie z kropką. Gdy cały wzorzec jest zwykłym tekstem, najprościej użyć `fixed = TRUE`.

In [4]:
koncowki <- c("dane.csv", "danexcsv", "dane.csv.bak")
grepl(".csv", koncowki, perl = TRUE)
grepl("\\.csv", koncowki, perl = TRUE)
grepl(".csv", koncowki, fixed = TRUE)

[1] TRUE TRUE TRUE
[1]  TRUE FALSE  TRUE
[1]  TRUE FALSE  TRUE


Pierwszy wzorzec pasuje do wszystkich trzech napisów. Dwa kolejne odrzucają `danexcsv`, ale nadal dopasowują fragment `dane.csv.bak`. Samo znalezienie `.csv` nie oznacza jeszcze końca nazwy.

**Zadanie R.4.** Dla `c("model+a", "modelaa", "model+a.txt")` znajdź dosłowny fragment `model+a` na dwa sposoby: przez `fixed = TRUE` i przez REGEX z zabezpieczonym plusem. Oba sposoby powinny wskazać dwie pozycje.

## 5. Kotwice: fragment czy cały napis?

`^` wskazuje początek, a `$` koniec tekstu; w trybie wielowierszowym mogą wskazywać granice wierszy. Użycie obu ogranicza dopasowanie do całego zapisu, zamiast dowolnego fragmentu. Dla ścisłej kontroli całego łańcucha w R/PCRE dostępne są także `\A` i `\z`; `\z` wymaga rzeczywistego końca, również gdy napis zawiera końcowy znak nowej linii.

In [5]:
kody <- c("AB-123", "XAB-123", "AB-1234", "AB-123\n")
grepl("AB-[0-9]{3}", kody, perl = TRUE)
grepl("\\AAB-[0-9]{3}\\z", kody, perl = TRUE)
grep("\\.csv$", koncowki, value = TRUE, perl = TRUE)

[1] TRUE TRUE TRUE TRUE
[1]  TRUE FALSE FALSE FALSE
[1] "dane.csv"


W drugim wyniku poprawny jest tylko pierwszy kod. Trzeci odrzuca dodatkową cyfrę, drugi dodatkowy prefiks, a czwarty znak końca linii. W trzecim poleceniu otrzymamy wyłącznie `dane.csv`.

W RegExr z silnikiem JavaScript użyj na potrzeby prostych nazw `^AB-[0-9]{3}$`. Jeżeli wklejasz wiele nazw w osobnych wierszach, włącz flagi `g` i `m`; `m` zmienia znaczenie `^` i `$` na granice wierszy. W R wektor zawiera osobne napisy, więc do takiego sprawdzania nie potrzeba trybu wielowierszowego.

**Zadanie R.5.** Wybierz wyłącznie nazwy postaci `raport_` + dwie cyfry + `.csv`. Sprawdź również `kopia_raport_01.csv`, `raport_01.csv.bak` i `raport_1.csv` — mają nie pasować. W R użyj `\A` i `\z`, a w RegExr przećwicz wariant `^...$`.

## 6. Klasy znaków i liczba powtórzeń

Klasa `[...]` wybiera jeden znak z podanego zestawu. Kwantyfikator mówi, ile razy ma wystąpić poprzedni element. Na przykład `[AB]{2}` pozwala na `AA`, `AB`, `BA` i `BB`, a nie wyłącznie na ciąg `AB`.

| Zapis | Znaczenie |
| :--- | :--- |
| `[0-9]` | jedna cyfra od 0 do 9 |
| `[ABC]` | jeden ze znaków A, B, C |
| `[^0-9]` | jeden znak inny niż cyfra z tego zakresu |
| `?` | zero lub jedno wystąpienie |
| `*` | zero lub więcej wystąpień |
| `+` | co najmniej jedno wystąpienie |
| `{3}` | dokładnie trzy wystąpienia |
| `{2,4}` | od dwóch do czterech wystąpień |

In [6]:
numery <- c("12-345", "1-345", "12345", "12-3456", "ab-345")
grepl("\\A[0-9]{2}-[0-9]{3}\\z", numery, perl = TRUE)

liczby_tekst <- c("12", "-12", "12,50", "-12,5", "12,", "")
grepl("\\A-?[0-9]+(,[0-9]{1,2})?\\z", liczby_tekst, perl = TRUE)

[1]  TRUE FALSE FALSE FALSE FALSE
[1]  TRUE  TRUE  TRUE  TRUE FALSE FALSE


W pierwszym przykładzie pasuje tylko `12-345`. W drugim znak minus jest opcjonalny, ale część całkowita obowiązkowa. Opcjonalna grupa po przecinku wymaga jednej lub dwóch cyfr, dlatego `12,` nie pasuje. Wzorzec kodu pocztowego sprawdza format, nie istnienie kodu w rejestrze.

**Zadanie R.6.** Zbuduj wzorzec identyfikatora: litera A albo B, myślnik, od dwóch do czterech cyfr. Przygotuj trzy poprawne i trzy błędne przykłady, w tym pusty napis. Sprawdź wszystkie jednym wywołaniem `grepl()`.

## 7. Alternatywa, grupowanie i całe słowa

Operator `|` oznacza alternatywę. Nawiasy łączą jej warianty we wspólną część wzorca. To ważne przy kotwicach: `^kot|pies$` oznacza „zaczyna się od kot **lub** kończy się na pies”, a nie dwa dopuszczalne całe napisy.

In [7]:
slowa <- c("kot", "pies", "kotek", "wilkopies", "kot i pies")
grepl("^kot|pies$", slowa, perl = TRUE)
grepl("\\A(kot|pies)\\z", slowa, perl = TRUE)
grepl("\\bkot\\b", slowa, perl = TRUE)

[1] TRUE TRUE TRUE TRUE TRUE
[1]  TRUE  TRUE FALSE FALSE FALSE
[1]  TRUE FALSE FALSE FALSE  TRUE


Pierwszy zapis dopasowuje wszystkie przykłady. Drugi tylko `kot` i `pies`. Trzeci wyszukuje słowo `kot` również wewnątrz zdania, ale pomija `kotek`. Granica `\b` zależy od tego, co silnik uznaje za znak słowa; poniższe przykłady celowo nie wymagają rozstrzygania granic przy polskich literach.

**Zadanie R.7.** Wybierz nazwy kończące się `.csv` lub `.xlsx` z `c("dane.csv", "dane.xlsx", "dane.csv.bak", "dane.txt")`. Pogrupuj alternatywę i dodaj kotwicę końca. Powinny pozostać dwie nazwy.

## 8. Wydobywanie dopasowań: pierwsze i wszystkie

`grepl()` odpowiada, czy wzorzec istnieje, ale nie zwraca jego treści. Do ekstrakcji użyjemy `regexpr()` lub `gregexpr()` wraz z `regmatches()`. Pierwsza funkcja wyszukuje pierwsze dopasowanie w każdym napisie, druga wszystkie. Ich wynik zawiera pozycje oraz długości, z których `regmatches()` wydobywa tekst.

In [8]:
opisy <- c("Zamówienia: AB-123 i CD-456", "Brak kodu", "Zwrot: EF-789")
wzorzec_kodu <- "[A-Z]{2}-[0-9]{3}"

pierwsze <- regexpr(wzorzec_kodu, opisy, perl = TRUE)
regmatches(opisy, pierwsze)

wszystkie <- regmatches(opisy, gregexpr(wzorzec_kodu, opisy, perl = TRUE))
wszystkie
lengths(wszystkie)

[1] "AB-123" "EF-789"
[[1]]
[1] "AB-123" "CD-456"

[[2]]
character(0)

[[3]]
[1] "EF-789"

[1] 2 0 1


Wynik pierwszej ekstrakcji nie zawiera osobnego elementu dla `Brak kodu`: otrzymasz dwa napisy. Wariant ze wszystkimi dopasowaniami zwraca listę trzech elementów, w tym pusty wektor dla drugiego opisu. **Nie przypisuj bez sprawdzenia krótszego wyniku do wierszy ramki.** Gdy potrzebujesz jednej wartości na wiersz, przygotuj wektor braków i uzupełnij tylko znalezione pozycje:

In [9]:
pierwszy_kod <- rep(NA_character_, length(opisy))
znaleziono <- pierwsze > 0
pierwszy_kod[znaleziono] <- regmatches(opisy, pierwsze)
data.frame(opis = opisy, pierwszy_kod = pierwszy_kod)

                         opis pierwszy_kod
1 Zamówienia: AB-123 i CD-456       AB-123
2                   Brak kodu         <NA>
3               Zwrot: EF-789       EF-789


**Zadanie R.8.** Z napisów `c("Cena 120 PLN, dostawa 15 PLN", "Brak opłat", "Cena 90 PLN")` wydobądź wszystkie ciągi cyfr. Zachowaj listę z jednym elementem na napis. Oczekiwane liczby dopasowań to 2, 0 i 1.

## 9. Zamiana tekstu i grupy przechwytujące

`sub()` zastępuje pierwsze dopasowanie w każdym napisie, a `gsub()` wszystkie. Nawiasy `(...)` zapamiętują dopasowane fragmenty. W argumencie zamiany odwołujemy się do nich w R przez `"\\1"`, `"\\2"` itd.

In [10]:
tekst <- "R---analiza---danych"
sub("-+", " ", tekst, perl = TRUE)
gsub("-+", " ", tekst, perl = TRUE)

daty_tekst <- c("2026-10-03", "2026-12-15", "brak")
wzorzec_daty <- "\\A([0-9]{4})-([0-9]{2})-([0-9]{2})\\z"
sub(wzorzec_daty, "\\3.\\2.\\1", daty_tekst, perl = TRUE)

[1] "R analiza---danych"
[1] "R analiza danych"
[1] "03.10.2026" "15.12.2026" "brak"      


Grupa 1 przechwytuje rok, 2 miesiąc, a 3 dzień. Wynik to `03.10.2026`, `15.12.2026` oraz niezmieniony `brak`. **Brak dopasowania w `sub()` pozostawia oryginalny napis**, więc sama zamiana nie potwierdza poprawności całej kolumny. REGEX kontroluje tutaj postać zapisu, nie poprawność kalendarzową: `2026-99-99` też spełnia wzorzec.

W narzędziu Replace w RegExr przy silniku JavaScript odpowiednikiem zamiany będzie `$3.$2.$1`. Nie kopiuj takiego zapisu bez zmiany do `sub()` w R.

**Zadanie R.9.** Zamień poprawnie zapisane identyfikatory `AB-123` i `CD-456` na `123/AB` oraz `456/CD`, korzystając z dwóch grup. Dodaj `brak` do danych i wyjaśnij, dlaczego pozostał niezmieniony.

## 10. Chciwość: dlaczego wzorzec znalazł za dużo?

Powtórzenia domyślnie próbują dopasować możliwie dużo tekstu. W `"A" oraz "B"` wzorzec zaczynający i kończący się cudzysłowem z `.*` pośrodku może objąć oba cytaty naraz. Znak `?` po kwantyfikatorze ogranicza chciwość. Często czytelniej określić, jakich znaków wewnątrz fragmentu **nie** dopuszczamy.

In [11]:
cytaty <- 'Wybrano "A" oraz "B".'
regmatches(cytaty, gregexpr('".*"', cytaty, perl = TRUE))
regmatches(cytaty, gregexpr('".*?"', cytaty, perl = TRUE))
regmatches(cytaty, gregexpr('"[^"]*"', cytaty, perl = TRUE))

[[1]]
[1] "\"A\" oraz \"B\""

[[1]]
[1] "\"A\"" "\"B\""

[[1]]
[1] "\"A\"" "\"B\""



Pierwszy wynik zawiera jeden długi fragment, dwa kolejne po dwa krótkie. Ostatni wzorzec oznacza cudzysłów, zero lub więcej znaków innych niż cudzysłów i cudzysłów zamykający. Przykład nie obsługuje zagnieżdżonych ani specjalnie zabezpieczonych cudzysłowów; złożone pliki CSV lub HTML czytaj odpowiednim parserem.

**Zadanie R.10.** Z napisu `"[cena: 120] [stan: nowy]"` wydobądź dwa osobne fragmenty w nawiasach kwadratowych. Porównaj wariant chciwy i niechciwy. Pamiętaj, że dosłowne nawiasy trzeba zabezpieczyć we wzorcu.

## 11. Sprawdzanie kontekstu: przykład dodatkowy

Czasem chcemy wydobyć liczbę tylko wtedy, gdy obok występuje konkretna jednostka. Konstrukcja `(?=...)` sprawdza to, co następuje dalej, ale nie dołącza tego fragmentu do wyniku. To dodatnie sprawdzenie wprzód, tzw. *lookahead*.

In [12]:
oferta <- "Cena 120 PLN; powierzchnia 45 m2; dostawa 20 PLN"
kwoty <- regmatches(oferta, gregexpr("[0-9]+(?= PLN)", oferta, perl = TRUE))
kwoty
as.numeric(kwoty[[1]])

[[1]]
[1] "120" "20" 

[1] 120  20


Otrzymasz liczby 120 i 20; jednostka `PLN` nie jest częścią wydobytego tekstu. To świadomie prosty wzorzec dla nieujemnych kwot całkowitych oddzielonych jedną spacją. Przy ułamkach i separatorach tysięcy trzeba zdefiniować inną postać liczby.

**Zadanie R.11.** Z `"czas: 15 min; dystans: 5 km; przerwa: 10 min"` wydobądź tylko liczby stojące przed ` min`. Oczekiwane wartości to 15 i 10.

## 12. Czyszczenie tekstu i filtrowanie ramki

Zanim zbudujesz bardziej złożony wzorzec, sprawdź spacje i braki. Poniżej porządkujemy wyłącznie nadmiar białych znaków oraz wielkość liter. W `grepl()` brak tekstu nie staje się zwykłym dopasowaniem, ale jawne `!is.na()` pozwala zapisać intencję i później oddzielnie policzyć braki.

In [13]:
zgloszenia <- data.frame(
  id = 1:5,
  temat = c("  BŁĄD   logowania ", "Pytanie o raport", NA,
             "błąd eksportu", "Prośba o dostęp")
)
zgloszenia$temat_czysty <- tolower(trimws(gsub("\\s+", " ", zgloszenia$temat, perl = TRUE)))
wybrane <- !is.na(zgloszenia$temat_czysty) &
  grepl("\\Abłąd(?: |\\z)", zgloszenia$temat_czysty, perl = TRUE)
zgloszenia[wybrane, c("id", "temat_czysty")]
sum(is.na(zgloszenia$temat))

  id   temat_czysty
1  1 błąd logowania
4  4  błąd eksportu
[1] 1


Wybrane zostają wiersze 1 i 4. Użyliśmy `(?:...)`, czyli grupy bez przechwytywania: porządkuje alternatywę „spacja lub koniec” i nie tworzy numeru do późniejszej zamiany. Polskie litery wpisujemy dosłownie; działanie konwersji wielkości liter zależy od prawidłowej konfiguracji kodowania i ustawień regionalnych R.

**Zadanie R.12.** Utwórz ramkę z sześcioma tytułami, w tym `NA`, nadmiarowymi spacjami i różną wielkością liter. Po oczyszczeniu wybierz tytuły zaczynające się od słowa `raport`, ale odrzuć `raportowanie`. Zachowaj identyfikatory i osobno policz braki.

## 13. Testowanie wzorca na przykładach i kontrprzykładach

Dobry pokaz zawiera tekst, który powinien pasować, oraz podobny, który powinien zostać odrzucony. Poniżej sprawdzamy format kodu. Wzorzec ma wymagać dokładnie dwóch wielkich liter ASCII, myślnika i trzech cyfr, bez dodatkowych znaków.

In [14]:
przypadki <- data.frame(
  tekst = c("AB-123", "XY-007", "A-123", "AB-12", "ab-123", "XAB-123", "AB-123 ", ""),
  oczekiwane = c(TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE)
)
wzorzec_scisly <- "\\A[A-Z]{2}-[0-9]{3}\\z"
przypadki$wynik <- grepl(wzorzec_scisly, przypadki$tekst, perl = TRUE)
przypadki$zgodnosc <- przypadki$wynik == przypadki$oczekiwane
przypadki
stopifnot(all(przypadki$zgodnosc))

    tekst oczekiwane wynik zgodnosc
1  AB-123       TRUE  TRUE     TRUE
2  XY-007       TRUE  TRUE     TRUE
3   A-123      FALSE FALSE     TRUE
4   AB-12      FALSE FALSE     TRUE
5  ab-123      FALSE FALSE     TRUE
6 XAB-123      FALSE FALSE     TRUE
7 AB-123       FALSE FALSE     TRUE
8              FALSE FALSE     TRUE


`stopifnot()` kończy się bez komunikatu, jeżeli wszystkie oczekiwania są spełnione, a zgłasza błąd, gdy test nie przechodzi. To sposób na sprawdzenie skutków zmiany wzorca. Dane brakujące testujemy osobno — nie utożsamiamy ich z pustym napisem `""` ani z błędnym kodem.

**Zadanie R.13.** Przygotuj co najmniej osiem testów dla nazwy `raport_` + dwie cyfry + `.csv`: dwa poprawne, zły prefiks, złą liczbę cyfr, błędną kropkę, dodatkową końcówkę, końcową spację i pusty napis. Zapisz oczekiwane wyniki przed uruchomieniem REGEX. Przenieś zwykłe przypadki także do trybu Tests w RegExr.

## 14. Pokaz łączący etapy: tekst oferty na dane

Mamy trzy fikcyjne oferty w ustalonym formacie. Najpierw sprawdzamy cały zapis, dopiero potem wydobywamy pola i konwertujemy kwotę. Akceptujemy wyłącznie całkowite kwoty w PLN; nierozpoznane wiersze zachowujemy z brakami w nowych kolumnach.

In [15]:
oferty <- data.frame(
  tekst = c("AB-123; cena=120 PLN", "CD-456; cena=85 PLN", "EF-789; cena=brak")
)
wzorzec_oferty <- "\\A([A-Z]{2}-[0-9]{3}); cena=([0-9]+) PLN\\z"
oferty$poprawny_format <- !is.na(oferty$tekst) &
  grepl(wzorzec_oferty, oferty$tekst, perl = TRUE)
oferty$kod <- NA_character_
oferty$cena <- NA_real_
ok <- oferty$poprawny_format
oferty$kod[ok] <- sub(wzorzec_oferty, "\\1", oferty$tekst[ok], perl = TRUE)
oferty$cena[ok] <- as.numeric(sub(wzorzec_oferty, "\\2", oferty$tekst[ok], perl = TRUE))
oferty

                 tekst poprawny_format    kod cena
1 AB-123; cena=120 PLN            TRUE AB-123  120
2  CD-456; cena=85 PLN            TRUE CD-456   85
3    EF-789; cena=brak           FALSE   <NA>   NA


W pierwszych dwóch wierszach otrzymujemy kod oraz cenę liczbową 120 i 85. Trzeci pozostaje do wyjaśnienia; nie został usunięty ani zamieniony na zerową cenę. Zachowanie oryginalnego tekstu ułatwia sprawdzenie transformacji.

**Zadanie R.14 — mały projekt.** Przygotuj sześć fikcyjnych opisów, np. `"ID=AB-123; metraz=45,5 m2"`. Uwzględnij trzy poprawne opisy, zły kod, brak liczby oraz `NA`.

1. Określ regułę formatu: kod jak wyżej, dodatnio zapisana liczba bez znaku z opcjonalną częścią po przecinku, jednostka `m2`.
2. Sprawdź zapis całego opisu. Następnie wydobądź kod i metraż tylko z pasujących wierszy.
3. W kopii wydobytej liczby zamień przecinek na kropkę i użyj `as.numeric()`.
4. Oddzielnie sprawdź, czy metraż jest większy od zera — zgodność zapisu i sens wartości to dwa różne warunki.
5. Pokaż ramkę z tekstem oryginalnym, rozpoznanym kodem, metrażem oraz statusem przetwarzania. Odróżnij brak danych od błędnego zapisu i od wartości niedodatniej.

Wskazówka: część liczbową można opisać przez `([0-9]+(?:,[0-9]+)?)`. Wewnętrzna grupa `(?:...)` nie zwiększa liczby grup przechwytujących. Zero spełni ten wzorzec i powinno zostać odrzucone dopiero przez kontrolę wartości.

## Ściąga: jakiego narzędzia potrzebuję?

| Pytanie | Funkcja lub konstrukcja |
| :--- | :--- |
| Które napisy pasują? | `grepl()` |
| Jakie są pozycje lub całe pasujące napisy? | `grep()`, `value = TRUE` |
| Jaki fragment znaleziono? | `regexpr()` i `regmatches()` |
| Jakie są wszystkie fragmenty w każdym napisie? | `gregexpr()` i `regmatches()` |
| Jak zmienić pierwsze lub wszystkie dopasowania? | `sub()` / `gsub()` |
| Jak szukać zwykłego tekstu bez REGEX? | `fixed = TRUE` |
| Jak sprawdzić cały napis w R/PCRE? | `\A...\z` z `perl = TRUE` |
| Jak zweryfikować własny wzorzec? | przykłady pozytywne, negatywne i przypadki graniczne |

**Zadanie R.15 — wyjaśnij na własnych przykładach.** Czym różni się wyszukanie fragmentu od sprawdzenia całego napisu? Dlaczego kropka wymaga uwagi? Dlaczego w kodzie R występują dwa ukośniki odwrotne? Kiedy ekstrakcja może zgubić zgodność z wierszami? Dlaczego poprawny format daty, kodu czy adresu e-mail nie potwierdza jego rzeczywistej poprawności?

W razie nieoczekiwanego wyniku skróć wzorzec do najmniejszej działającej części. Następnie dodawaj po jednym elemencie i sprawdzaj kontrprzykłady. Przy e-mailach traktuj prosty REGEX jako filtr o ustalonym zakresie, nie jako pełne potwierdzenie poprawności lub istnienia adresu.

## Materiały do dalszych ćwiczeń

- [RegExr](https://regexr.com/) — interaktywne dopasowania, objaśnienia składni oraz własne testy.
- [Regular Expressions as used in R](https://stat.ethz.ch/R-manual/R-devel/library/base/html/regex.html) — składnia i różnice między trybami dopasowania w R.
- [Pattern Matching and Replacement](https://stat.ethz.ch/R-manual/R-devel/library/base/html/grep.html) — argumenty i wyniki funkcji wyszukiwania oraz zamiany. Podrozdział używa własnych danych i pokazów dydaktycznych.

[← Wróć do tabel opisowych z gtsummary](rozdzial-2-gtsummary.ipynb)